<a href="https://colab.research.google.com/github/satyam-1605/RAG-from-basic-to-advance/blob/main/Rag_with_gemini_and_huggingface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install weaviate-client tiktoken pypdf rapidocr-onnxruntime

In [ ]:
!pip install langchain

In [ ]:
WEAVIATE_CLUSTER=""
WEAVIATE_API_KEY="API_KEY"


In [ ]:
from langchain_community.vectorstores import Weaviate
import os
import weaviate
from weaviate.classes.init import Auth

# Best practice: store your credentials in environment variables
# Corrected: Use the Python variables directly instead of os.environ
weaviate_url = WEAVIATE_CLUSTER
weaviate_api_key = WEAVIATE_API_KEY

# Connect to Weaviate Cloud
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=Auth.api_key(weaviate_api_key),
)

print(client.is_ready())

In [ ]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [ ]:
!pip install sentence-transformers

In [ ]:
!pip install langchain-huggingface

In [ ]:
# specify embedding model (using huggingface sentence transformer)
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model_name = "sentence-transformers/all-mpnet-base-v2"
#model_kwargs = {"device": "cuda"}
embeddings = HuggingFaceEmbeddings(
  model_name=embedding_model_name,
  #model_kwargs=model_kwargs
)

In [ ]:

from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("/content/RAG.pdf", extract_images=True)
pages = loader.load()

In [ ]:
pages

In [ ]:
!pip install langchain-text-splitters

In [ ]:

# Split text into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
docs = text_splitter.split_documents(pages)

In [ ]:
docs

In [ ]:
!pip install  langchain-weaviate

In [ ]:
import weaviate
from langchain_weaviate import WeaviateVectorStore
for doc in docs:
    doc.metadata = {k.replace(".", "_"): v for k, v in doc.metadata.items()}
vector_db = WeaviateVectorStore.from_documents(
    docs, embeddings, client=client, by_text=False
)

In [ ]:
results =vector_db.similarity_search("what is rag?", k=3)
for doc in results:
    print(doc.page_content)

In [ ]:


print(vector_db.similarity_search("what is rag?", k=3)[1].page_content)

In [ ]:


print(
    vector_db.similarity_search(
        "what is attention?", k=3)
    )

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [ ]:
prompt=ChatPromptTemplate.from_template(template)

In [ ]:
prompt

In [ ]:
!pip install langchain-huggingface

In [ ]:
from google.colab import userdata
google_api_token = userdata.get('GEMINI_API_KEY')

In [ ]:
!pip install langchain-google-genai

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(
    google_api_key=google_api_token,
    model="gemini-2.5-flash",
    max_tokens=5000,
    temperature=0.7,
)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


In [ ]:

output_parser=StrOutputParser()


In [ ]:
retriever=vector_db.as_retriever()

In [ ]:



rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | model
    | output_parser
)


In [ ]:
print(rag_chain.invoke("How does the RAG model differ from traditional language generation models?"))